In [1]:
from pathlib import Path
import os, html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILE = "Water Quality.csv"  # Change to full path if needed
OUT = Path("task3_storytelling_outputs")
OUT.mkdir(exist_ok=True)
if not Path(FILE).exists():
    raise FileNotFoundError(f"Cannot find {FILE!r}; set FILE to the full CSV path.")
df = pd.read_csv(FILE, low_memory=False)
required = {"ActivityStartDate", "CharacteristicName", "ResultValue"}
if required - set(df.columns):
    raise ValueError(f"Missing columns: {required-set(df.columns)}")
df["ActivityStartDate"] = pd.to_datetime(df["ActivityStartDate"], errors="coerce")
df["ResultValue"] = pd.to_numeric(df["ResultValue"], errors="coerce")
df = df.dropna(subset=["ActivityStartDate", "CharacteristicName", "ResultValue"])
if df.empty: raise ValueError("No valid date/numeric measurements found.")
df["Year"] = df.ActivityStartDate.dt.year
coverage = {"records":len(df), "parameters":df.CharacteristicName.nunique(),
            "locations":df.MonitoringLocationID.nunique() if "MonitoringLocationID" in df else "not available",
            "start":str(df.ActivityStartDate.min().date()), "end":str(df.ActivityStartDate.max().date())}
top = df.CharacteristicName.value_counts().head(8).index
yearly_rows, trend_rows = [], []
for param in top:
    sub = df[df.CharacteristicName == param]
    yearly = sub.groupby("Year").ResultValue.median()
    for year, val in yearly.items():
        yearly_rows.append({"CharacteristicName":param, "Year":int(year),
                            "median_ResultValue":val, "measurements":int((sub.Year==year).sum())})
    monthly = sub.set_index("ActivityStartDate").ResultValue.resample("MS").median().dropna()
    if len(monthly) >= 2:
        x = np.arange(len(monthly)); slope, intercept = np.polyfit(x, monthly.values, 1)
        trend_rows.append({"CharacteristicName":param, "monthly_points":len(monthly),
                           "start_month":str(monthly.index.min().date()), "end_month":str(monthly.index.max().date()),
                           "linear_slope_per_month_step":slope,
                           "direction":"upward" if slope>0 else "downward" if slope<0 else "flat"})
        fig, ax = plt.subplots(figsize=(9,4))
        ax.plot(monthly.index, monthly.values, marker="o", label="Monthly median")
        ax.plot(monthly.index, intercept+slope*x, "--", label="Linear trend")
        ax.set(title=f"{param}: monthly median", xlabel="Date", ylabel="ResultValue (original units)")
        ax.legend(); fig.tight_layout()
        safe = "".join(c if c.isalnum() else "_" for c in str(param))[:50]
        fig.savefig(OUT/f"timeseries_{safe}.png", dpi=140); plt.close(fig)
yearly_df, trend_df = pd.DataFrame(yearly_rows), pd.DataFrame(trend_rows)
yearly_df.to_csv(OUT/"yearly_parameter_summary.csv", index=False)
trend_df.to_csv(OUT/"linear_trend_summary.csv", index=False)
trend_table = trend_df.to_html(index=False, escape=True, border=0) if not trend_df.empty else "<p>Insufficient time coverage for trend estimates.</p>"
year_table = yearly_df.to_html(index=False, escape=True, border=0) if not yearly_df.empty else "<p>No yearly summary.</p>"
report = f"""<!doctype html><html><head><meta charset="utf-8"><title>AquaInsight Evidence Report</title>
<style>body{{font-family:Arial;max-width:1050px;margin:35px auto;line-height:1.55;color:#20242a}}h1,h2{{color:#164e63}}
table{{border-collapse:collapse;width:100%;font-size:14px}}th,td{{border:1px solid #ddd;padding:8px;text-align:left}}th{{background:#e7f2f5}}
.note{{background:#fff8e6;padding:12px;border-left:4px solid #d39b24}}</style></head><body>
<h1>AquaInsight: Water Quality Evidence Report</h1>
<p><b>Purpose:</b> summarize recorded measurements, temporal patterns and monitoring coverage.</p>
<h2>1. Dataset overview</h2><ul><li>Valid numeric records: {coverage['records']:,}</li>
<li>Distinct parameters: {coverage['parameters']}</li><li>Monitoring locations: {coverage['locations']}</li>
<li>Date coverage: {html.escape(coverage['start'])} to {html.escape(coverage['end'])}</li></ul>
<p>Rows with invalid dates or non-numeric measurements were excluded from these summaries.</p>
<h2>2. Temporal patterns</h2><p>For frequently recorded parameters, monthly medians and simple linear trend lines were calculated. Slopes are descriptive and do not prove significance or causality.</p>
{trend_table}<h2>3. Yearly medians</h2>{year_table}
<h2>4. Recommended next steps</h2><ul>
<li>Confirm apparent trends with Mann–Kendall and Sen's slope; consider seasonality and autocorrelation.</li>
<li>Check units, detection limits, laboratory methods and sampling coverage before comparing sites.</li>
<li>Use regulatory or domain-specific thresholds before calling a value a contamination hotspot.</li>
<li>Define a validated Water Quality Index formula before making an overall WQI claim.</li></ul>
<h2>5. Limitations</h2><div class="note">This report is descriptive. The source dataset does not inherently supply a validated Water Quality Index, potability outcome, urban/rural class or regulatory exceedance label. Different parameters may have different units. Sampling density is not contamination intensity.</div>
</body></html>"""
(OUT/"aquainsight_storytelling_report.html").write_text(report, encoding="utf-8")
print("Saved HTML report, summary CSVs and plots in", OUT.resolve())


Saved HTML report, summary CSVs and plots in C:\Users\YASH\Desktop\BrainyBeam\task 2 web\task3_storytelling_outputs
